# 02 — Leakage-Free Feature Engineering & Cutoff Cohorts (`P1-B2`)

Builds and verifies all 6 real-data feature Parquet tables (`operational` & `common` x `T=14, 21, 28`) from `data/oulad.duckdb`, checks all 4 anti-leakage invariants directly on the real dataset (`AUD-01`, `AUD-02`), and displays cohort statistics.

In [1]:
import json

import pandas as pd
from IPython.display import display

from oulad.config import DB_PATH, PROCESSED_DIR, REPORTS_DIR
from oulad.db import get_connection
from oulad.features import FORBIDDEN_FEATURE_COLS, export_all_feature_parquets

con = get_connection(DB_PATH, read_only=True)
summary = export_all_feature_parquets(con, out_dir=PROCESSED_DIR)
con.close()

summary_df = pd.DataFrame(summary)
display(summary_df)

committed_summary = json.loads((REPORTS_DIR / "cohort_summary.json").read_text(encoding="utf-8"))
assert committed_summary == summary
print("Exported 6 Parquet files and verified 100% exact match with reports/cohort_summary.json.")

,mode,cutoff,n_rows,n_features,positive_rate,top_activities,path
0,operational,14,28062,35,0.4526,"[forumng, homepage, oucontent, subpage, quiz]",features_op_c14.parquet
1,common,14,27491,35,0.4413,"[forumng, homepage, oucontent, subpage, quiz]",features_common_c14.parquet
2,operational,21,27880,36,0.4486,"[forumng, homepage, oucontent, subpage, quiz]",features_op_c21.parquet
3,common,21,27491,36,0.4413,"[forumng, homepage, oucontent, subpage, quiz]",features_common_c21.parquet
4,operational,28,27528,37,0.4414,"[forumng, homepage, oucontent, subpage, quiz]",features_op_c28.parquet
5,common,28,27491,37,0.4413,"[forumng, homepage, oucontent, subpage, quiz]",features_common_c28.parquet


Exported 6 Parquet files and verified 100% exact match with reports/cohort_summary.json.


In [2]:
# Verify anti-leakage & schema invariants directly on all 6 real-data Parquets
checks = []
for item in summary:
    mode = item["mode"]
    cutoff = item["cutoff"]
    df = pd.read_parquet(PROCESSED_DIR / item["path"])
    max_allowed_reg = cutoff if mode == "operational" else 14

    has_forbidden = any(c in df.columns for c in FORBIDDEN_FEATURE_COLS)
    reg_ok = bool(
        df["date_registration"].notna().all()
        and (df["date_registration"] < max_allowed_reg).all()
    )
    valid_rates = df["submission_rate_due"].dropna()
    rate_ok = bool(((valid_rates >= 0.0) & (valid_rates <= 1.0)).all())
    week_ok = (
        (("clicks_w2" in df.columns) == (cutoff >= 21))
        and (("clicks_w3" in df.columns) == (cutoff >= 28))
    )

    assert not has_forbidden and reg_ok and rate_ok and week_ok
    checks.append(
        {
            "file": item["path"],
            "mode": mode,
            "cutoff": cutoff,
            "max_date_reg": int(df["date_registration"].max()),
            "max_allowed_reg": max_allowed_reg,
            "submission_rate_min": round(float(valid_rates.min()), 4),
            "submission_rate_max": round(float(valid_rates.max()), 4),
            "invariants_pass": True,
        }
    )

display(pd.DataFrame(checks))

,file,mode,cutoff,max_date_reg,max_allowed_reg,submission_rate_min,submission_rate_max,invariants_pass
0,features_op_c14.parquet,operational,14,13,14,0.0,1.0,True
1,features_common_c14.parquet,common,14,13,14,0.0,1.0,True
2,features_op_c21.parquet,operational,21,20,21,0.0,1.0,True
3,features_common_c21.parquet,common,21,13,14,0.0,1.0,True
4,features_op_c28.parquet,operational,28,27,28,0.0,1.0,True
5,features_common_c28.parquet,common,28,13,14,0.0,1.0,True
